In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
from sklearn.decomposition import PCA
from pyriemann.utils.mean import mean_logeuclid
from pyriemann.utils.tangentspace import tangent_space

In [2]:
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 

In [3]:
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [4]:
def compute_enhanced_cst_pca(bar_s_source, bar_t_target_align, y_source, y_target_align, classes, n_clusters=3):
    """
    Enhance Cst computation using PCA-based clustering as described in section 2.4 of the paper.
    For each class:
    1. Train PCA on source data
    2. Project both source and target data onto PCA space
    3. For each principal component, create clusters based on component values
    4. Compute mean of each cluster as anchor points
    """
    
    # Lists to store all anchor points
    s_anchors = []
    t_anchors = []
    
    # For each class, create clusters using PCA
    for cls in classes:
        # Get data for this class
        s_cls = bar_s_source[y_source == cls]
        t_cls = bar_t_target_align[y_target_align == cls]
        
        if len(s_cls) < n_clusters or len(t_cls) < n_clusters:
            # If not enough samples, just use class mean
            s_anchors.append(np.mean(s_cls, axis=0))
            t_anchors.append(np.mean(t_cls, axis=0))
            continue
        
        # Train PCA on source data (just first component as shown in Figure 2)
        pca = PCA(n_components=1)
        pca.fit(s_cls)
        
        # Project both source and target data
        s_proj = pca.transform(s_cls).flatten()
        t_proj = pca.transform(t_cls).flatten()
        
        # Sort by PCA scores
        s_sorted_idx = np.argsort(s_proj)
        t_sorted_idx = np.argsort(t_proj)
        
        # Split into n_clusters clusters along the first principal component
        s_chunk_size = len(s_sorted_idx) // n_clusters
        t_chunk_size = len(t_sorted_idx) // n_clusters
        
        # Create clusters and compute means
        for i in range(n_clusters):
            s_start = i * s_chunk_size
            s_end = None if i == n_clusters-1 else (i+1) * s_chunk_size
            s_cluster_idx = s_sorted_idx[s_start:s_end]
            
            t_start = i * t_chunk_size
            t_end = None if i == n_clusters-1 else (i+1) * t_chunk_size
            t_cluster_idx = t_sorted_idx[t_start:t_end]
            
            # Compute and store means for this cluster
            s_anchors.append(np.mean(s_cls[s_cluster_idx], axis=0))
            t_anchors.append(np.mean(t_cls[t_cluster_idx], axis=0))
    
    # Stack all anchor points
    bar_S = np.column_stack(s_anchors)  # Shape: (n_features, n_anchors)
    bar_T = np.column_stack(t_anchors)  # Shape: (n_features, n_anchors)
    
    # Compute cross-product matrix (as in the paper)
    C_st = bar_S @ bar_T.T
    
    return C_st, bar_S, bar_T


In [5]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [6]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [7]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 3.5  # This gives 4.0 seconds.

In [10]:
def twofour_crosssession(n_classes):

    if(n_classes==2):
        event_ids = active_lr_event_ids
    else:
        event_ids = active_all_event_ids
    

    train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
    train_active_y = []         # List to hold event labels per subject.
    train_active_metadata = []  # List to hold event metadata per subject.

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
        
        # Read the GDF file (using preload=True to load data into memory).
        train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        train_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        train_active_events, _ = mne.events_from_annotations(train_raw, event_id=event_ids)
        
        # Create epochs from tmin to tmax.
        train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=event_ids, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        train_active_data = train_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(train_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = train_active_data.shape
        train_active_filtered_data = np.empty_like(train_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    train_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=5     # Lower order for a smoother causal filter.
                )
        # Apply the causal bandpass filter channel‐wise for each trial.
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         train_active_filtered_data[trial, ch, :] = train_active_data[trial, ch, :]
        
        # Append the processed data, labels, and event metadata.
        train_active_X.append(train_active_filtered_data)
        train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
        train_active_metadata.append(train_active_epochs.events)

    print("Loaded data for", len(train_active_X), "subjects.")

    eval_active_X = []         
    eval_active_y = []        
    eval_active_metadata = [] 

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
        mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
        true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
        # Read the GDF file (using preload=True to load data into memory).
        eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        eval_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
        
        # Create epochs from tmin to tmax.
        eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        eval_active_data = eval_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(eval_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = eval_active_data.shape
        eval_active_filtered_data = np.empty_like(eval_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    eval_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=5     # Lower order for a smoother causal filter.
                )
        
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         eval_active_filtered_data[trial, ch, :] = eval_active_data[trial, ch, :]
                    
        
        # Append the processed data, labels, and event metadata.
        eval_active_X.append(eval_active_filtered_data)
        eval_active_y.append(true_y)  # The third column holds the event code.
        eval_active_metadata.append(eval_active_epochs.events)

    print("Loaded data for", len(eval_active_X), "subjects.")

    if(n_classes==2):
        eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
        eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]
    
    train_active_y = encode_labels(train_active_y)
    eval_active_y = encode_labels(eval_active_y)

    if(n_classes==2):
        classes = [0, 1]
    else:
        classes = [0, 1, 2, 3]

    align_per_class = 14
    n_subjects = 9
    
    accuracies = []

    for subj_idx in range(len(train_active_X)):
        # print(subj_idx)
        # Split data into train/test using leave-one-subject-out
        X_target = eval_active_X[subj_idx]
        y_target = eval_active_y[subj_idx]
        
        # Concatenate data from other subjects
        X_source = train_active_X[subj_idx]
        y_source = train_active_y[subj_idx]

            # Split target data into alignment and test sets
        align_indices = []
        test_indices = []
        for cls in classes:
            cls_indices = np.where(y_target == cls)[0]
            np.random.shuffle(cls_indices)
            align_indices.extend(cls_indices[:align_per_class])
            test_indices.extend(cls_indices[align_per_class:])
        
        X_target_align = X_target[align_indices]  # Shape: (56, 22, 1001)
        y_target_align = y_target[align_indices]  # Shape: (56,)
        X_target_test = X_target[test_indices]    # Shape: (232, 22, 1001)
        y_target_test = y_target[test_indices]    # Shape: (232,)


        print("Source labels:", np.unique(y_source, return_counts=True))
        print("Target test labels:", np.unique(y_target_test, return_counts=True))
        print("Target align labels:", np.unique(y_target_align, return_counts=True))

        # Compute covariance matrices
        n_samples = X_source.shape[2]  # 1001
        print(n_samples)

        cov_estimator = Covariances(estimator='scm')  # Sample Covariance Matrix estimator
        C_source = cov_estimator.fit_transform(X_source)          # Shape: (288, 22, 22)
        C_target_align = cov_estimator.fit_transform(X_target_align)  # Shape: (56, 22, 22)
        C_target_test = cov_estimator.fit_transform(X_target_test)    # Shape: (232, 22, 22)

        # After computing C_source, check symmetry
        symmetry_error = np.max(np.abs(C_source - np.transpose(C_source, (0, 2, 1))))
        print(f"Symmetry error: {symmetry_error}")

        # # Compute covariance matrices with regularization
        M_source = mean_logeuclid(C_source)        # Shape: (22, 22)
        M_target_align = mean_logeuclid(C_target_align)
        M_target_test = mean_logeuclid(C_target_test)

        S_source = tangent_space(C_source, M_source, metric='logeuclid')          # Shape: (288, 253)
        S_target_align = tangent_space(C_target_align, M_target_align, metric='logeuclid')
        S_target_test = tangent_space(C_target_test, M_target_test, metric='logeuclid')
        
        # # Rescale to average norm of 1
        average_norm_source = np.mean(np.linalg.norm(S_source, axis=1))
        print(average_norm_source)
        bar_s_source = S_source  / average_norm_source  # Shape: (288, 253)

        average_norm_target_align = np.mean(np.linalg.norm(S_target_align, axis=1))
        bar_t_target_align = S_target_align / average_norm_target_align   # Shape: (56, 253)

        average_norm_target_test = np.mean(np.linalg.norm(S_target_test, axis=1))
        bar_t_target_test = S_target_test / average_norm_target_test  # Shape: (232, 253)

        # # Compute class means for alignment
        bar_s_k = [np.mean(bar_s_source[y_source == k], axis=0) for k in classes]  # List of 2 vectors, each (253,)
        bar_t_k = [np.mean(bar_t_target_align[y_target_align == k], axis=0) for k in classes]  # List of 2 vectors, each (253,)


        # # Stack class means into matrices
        bar_S = np.column_stack(bar_s_k)  # Shape: (253, 2)
        bar_T = np.column_stack(bar_t_k)  # Shape: (253, 2)


        # # Compute cross-product matrix
        C_st = bar_S @ bar_T.T  # Shape: (253, 253)

        # C_st, bar_S, bar_T = compute_enhanced_cst_pca(bar_s_source, bar_t_target_align, y_source, y_target_align, classes, n_clusters=3)

        # # print(np.isnan(C_st).sum(), np.isinf(C_st).sum())  # Check for numerical issues

        # # PCA Augmentation Step (as per the paper)
        # # 1. Perform SVD on C_st
        U, D, Vh = np.linalg.svd(C_st, full_matrices=False)  # U: (253, 253), D: (253,), Vh: (253, 253)
        # U contains left singular vectors, D contains singular values, Vh is V^T (right singular vectors transposed)

        cumsum_D = np.cumsum(D)
        total_D = np.sum(D)
        N_v = np.searchsorted(cumsum_D, 0.999 * total_D) + 1
        if N_v == 0:
            N_v = 1  # Ensure at least one singular vector

        # Compute rotation matrix
        rotation_matrix = U[:, :N_v] @ Vh[:N_v, :]  # Shape: (253, 253)
        # Align target test vectors
        hat_t_test = bar_t_target_test @ rotation_matrix
        

        # # Train SVC on source data
        clf = SVC(kernel='linear')
        clf.fit(bar_s_source, y_source)

        # # Predict on aligned test data
        y_pred = clf.predict(hat_t_test)


        # # After prediction, check distribution of classes
        # # Compute accuracy
        acc = accuracy_score(y_target_test, y_pred)
        accuracies.append(acc)

        
    for subj_idx in range(len(train_active_X)):
        print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

    print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
    

        
    

In [11]:
twofour_crosssession(2)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 1: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 2: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 751)
Loaded data for 9 subjects.
Source labels: (array([0, 1]), array([72, 72]))
Target test labels: (array([0, 1]), array([58, 58]))
Target align labels: (array([0, 1]), array([14, 14]))
751
Symmetry error: 0.0
2.735278757102966
Source labels: (array([0, 1]), array([72, 72]))
Target test labels: (array([0, 1]), array([58, 58]))
Target align labels: (array([0, 1]), array([14, 14]))
751
Symmetry error: 0.0
2.5922441867816235
Source labels: (array([0, 1]), array([72, 72]))
Target test labels: (array([0, 1]), array([58, 58]))
Target align labels: (array([0, 1]), array([14, 14]))
751
Symmetry error: 0.0
3.160852071602046
Source labels: (array([0, 1]), array([72, 72]))
Target test labels: (array([0, 1]), array([58, 58]))
Target align labels: (array([0, 1]), array([14, 14]))
751
Symmetry error: 0.0
2.678728051809609
Source labels: (array([0

In [12]:
twofour_crosssession(4)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 1: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 2: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 751)
Loaded data for 9 subjects.
Source labels: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Target test labels: (array([0, 1, 2, 3]), array([58, 58, 58, 58]))
Target align labels: (array([0, 1, 2, 3]), array([14, 14, 14, 14]))
751
Symmetry error: 0.0
2.795809821605153
Source labels: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Target test labels: (array([0, 1, 2, 3]), array([58, 58, 58, 58]))
Target align labels: (array([0, 1, 2, 3]), array([14, 14, 14, 14]))
751
Symmetry error: 0.0
2.5827643246751384
Source labels: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Target test labels: (array([0, 1, 2, 3]), array([58, 58, 58, 58]))
Target align labels: (array([0, 1, 2, 3]), array([14, 14, 14, 14]))
751
Symmetry error: 0.0
3.147756397515045
Source labels: (array([0, 1, 2, 3]), array([72, 72, 72, 72]))
Target test labels: (array([0, 1,